# RF-DETR Multi-Class Prediction & PaddleOCR Pipeline
### Post-Training Inference on Images with Recalibrated Bounding Boxes & PaddleOCR

This notebook runs end-to-end inference on warehouse images following model training:
1. **Model Loading**: Automatically prioritizes the best precision checkpoint (`rfdetr_best_precision.pth` / `best_model_precision.pth`).
2. **Bounding-Box Recalibration**:
   - **Boundary-Box Shrinkage**: Contracts loose query boundaries towards center (`shrink_factor=0.96`), eliminating shelf border slack.
   - **Bounding-Box Averaging**: Fuses overlapping duplicate candidate queries (IoU >= 0.50) via confidence-weighted coordinate averaging.
3. **PaddleOCR Text Recognition**: Crops tightly recalibrated tag patches and extracts alphanumeric location codes (e.g. `"LOC-A-12-04"`).
4. **Structured Results**: Exports structured detections (boxes, classes, OCR texts, confidences) to JSON.
5. **High-Contrast Visualization**: Saves annotated images with bounding boxes and text badges to `predictions/`.

In [ ]:
# STEP 0: Install Dependencies
import sys

%pip install -q "pycocotools" "pillow>=10.0.0" "numpy>=1.24.0" "matplotlib>=3.7.0" "paddlepaddle>=2.6.0" "paddleocr>=2.7.0" "opencv-python-headless>=4.9.0.80" "torchvision" "tqdm"

try:
    import rfdetr
    print("✓ rfdetr is already installed.")
except ImportError:
    print("Installing rfdetr...")
    %pip install -q "rfdetr"

print("✓ Prediction environment dependencies configured.")

In [ ]:
# CELL 1: Imports, Logger & Device Configuration
import os
import sys
import io
import json
import time
import logging
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as patches

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("RFDETR_Prediction")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Inference Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

OUTPUT_DIR = Path("predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# CELL 2: Discover and Load Fine-Tuned Model Checkpoint
# Resilient workspace & pipeline root resolver
REPO_ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("../..")] if (p / "coco_files").exists()), Path("..").resolve())
PIPELINE_ROOT = next((p.resolve() for p in [Path("."), Path(".."), Path("../..")] if (p / "04_training").exists()), Path(".").resolve())
CANDIDATE_CKPTS = [
    PIPELINE_ROOT / "04_training" / "model" / "best_model_clean.pth",
    REPO_ROOT / "model" / "best_model_clean.pth",
    REPO_ROOT / "multi_class_train_rfdetr" / "model" / "best_model_full_data.pth",
    REPO_ROOT / "best_model_full_data.pth",
    REPO_ROOT / "best_model.pth"
]

CKPT_PATH = None
for p in CANDIDATE_CKPTS:
    if p.exists() and p.stat().st_size > 1024 * 1024:
        CKPT_PATH = p
        break

print(f"Selected Checkpoint: {CKPT_PATH}")

MODEL_FAMILY = "RFDETRBase"
RESOLUTION = 1008
CLASSES = ["location_tag", "Blue_aisle", "blue_bay"]

def load_inference_model(ckpt_path: Optional[Path]):
    try:
        from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium, RFDETRBase, RFDETRLarge
        cls_map = {
            "rfdetrnano": RFDETRNano,
            "rfdetrsmall": RFDETRSmall,
            "rfdetrmedium": RFDETRMedium,
            "rfdetrbase": RFDETRBase,
            "rfdetrlarge": RFDETRLarge
        }
        ModelClass = cls_map.get(MODEL_FAMILY.lower(), RFDETRBase)
        
        if hasattr(ModelClass, "maybe_download_pretrain_weights"):
            ModelClass.maybe_download_pretrain_weights = lambda self: None
        if hasattr(ModelClass, "load_pretrain_weights"):
            ModelClass.load_pretrain_weights = lambda self: None
            
        wrapper = ModelClass(num_classes=len(CLASSES), resolution=RESOLUTION, pretrain_weights=None)
        
        if hasattr(wrapper, "model") and hasattr(wrapper.model, "model") and isinstance(wrapper.model.model, nn.Module):
            model = wrapper.model.model
        elif hasattr(wrapper, "model") and isinstance(wrapper.model, nn.Module):
            model = wrapper.model
        else:
            model = wrapper
            
        if ckpt_path and ckpt_path.exists():
            print(f"Loading weights from {ckpt_path}...")
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            state = ckpt.get("model", ckpt)
            
            for k in ["class_embed.weight", "model.class_embed.weight", "module.class_embed.weight"]:
                if k in state:
                    n_cls = state[k].shape[0]
                    if hasattr(model, "class_embed") and model.class_embed.out_features != n_cls:
                        model.class_embed = nn.Linear(model.class_embed.in_features, n_cls)
                    break
                    
            filtered = {k.replace("model.", "").replace("module.", ""): v for k, v in state.items()}
            cur = model.state_dict()
            matched = {k: v for k, v in filtered.items() if k in cur and cur[k].shape == v.shape}
            model.load_state_dict(matched, strict=False)
            print(f"✓ Successfully loaded {len(matched)} / {len(cur)} tensor layers.")
            
        model.to(device).eval()
        return model
    except Exception as e:
        print(f"Notice during model loading: {e}")
        return None

rfdetr_model = load_inference_model(CKPT_PATH)
if rfdetr_model is not None:
    print("✓ RF-DETR model loaded and ready for inference.")

In [ ]:
# CELL 3: Initialize PaddleOCR Engine
ocr_engine = None
try:
    from paddleocr import PaddleOCR
    print("Initializing PaddleOCR engine (English, angle classification)...")
    ocr_engine = PaddleOCR(use_angle_cls=True, lang="en", use_gpu=torch.cuda.is_available(), show_log=False)
    print("✓ PaddleOCR engine initialized successfully.")
except Exception as e:
    print(f"Notice: PaddleOCR could not be initialized ({e}). Predictions will run detection only.")

In [ ]:
# CELL 4: End-to-End Prediction Function with Bounding-Box Averaging & Shrinkage Recalibration

def recalibrate_and_average_boxes(
    boxes: list,
    scores: list,
    class_ids: list,
    enable_averaging: bool = True,
    enable_shrinkage: bool = True,
    shrink_factor: float = 0.96,
    fusion_iou: float = 0.50,
    max_w: float = 1e6,
    max_h: float = 1e6
) -> list:
    """
    Recalibrate detections via:
    1. Boundary-Box Shrinkage: Contracts loose query boundaries towards center by shrink_factor
       (e.g., 0.96 shrinks width and height by 4%), eliminating background margin jitter
       and tightening crops for downstream PaddleOCR.
    2. Bounding-Box Averaging: Clusters overlapping queries of the same class (IoU >= fusion_iou)
       and performs confidence-weighted coordinate averaging, suppressing duplicate false alarms.
    """
    if not boxes or len(boxes) == 0:
        return []
    
    # 1. Boundary-box shrinkage
    shrunk = []
    for i, b in enumerate(boxes):
        if enable_shrinkage and shrink_factor < 1.0:
            cx = (b[0] + b[2]) / 2.0
            cy = (b[1] + b[3]) / 2.0
            w = (b[2] - b[0]) * shrink_factor
            h = (b[3] - b[1]) * shrink_factor
            x1 = max(0.0, cx - w / 2.0)
            y1 = max(0.0, cy - h / 2.0)
            x2 = min(float(max_w), cx + w / 2.0)
            y2 = min(float(max_h), cy + h / 2.0)
            shrunk_b = [x1, y1, x2, y2]
        else:
            shrunk_b = list(b)
        
        shrunk.append({
            "box": shrunk_b,
            "score": float(scores[i]),
            "class_id": int(class_ids[i])
        })
        
    if not enable_averaging or len(shrunk) <= 1:
        return shrunk
        
    # 2. Bounding-box weighted averaging (WBF)
    fused = []
    distinct_classes = set(p["class_id"] for p in shrunk)
    for cid in distinct_classes:
        c_preds = [p for p in shrunk if p["class_id"] == cid]
        c_preds.sort(key=lambda x: x["score"], reverse=True)
        
        clusters = []
        for p in c_preds:
            matched = False
            for cl in clusters:
                b1, b2 = p["box"], cl[0]["box"]
                xA, yA = max(b1[0], b2[0]), max(b1[1], b2[1])
                xB, yB = min(b1[2], b2[2]), min(b1[3], b2[3])
                inter = max(0.0, xB - xA) * max(0.0, yB - yA)
                a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
                a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
                iou = inter / max(1e-7, a1 + a2 - inter)
                if iou >= fusion_iou:
                    cl.append(p)
                    matched = True
                    break
            if not matched:
                clusters.append([p])
                
        for cl in clusters:
            if len(cl) == 1:
                fused.append(cl[0])
            else:
                weights = [max(0.01, c["score"]) for c in cl]
                total_w = sum(weights)
                avg_x1 = sum(c["box"][0] * w for c, w in zip(cl, weights)) / total_w
                avg_y1 = sum(c["box"][1] * w for c, w in zip(cl, weights)) / total_w
                avg_x2 = sum(c["box"][2] * w for c, w in zip(cl, weights)) / total_w
                avg_y2 = sum(c["box"][3] * w for c, w in zip(cl, weights)) / total_w
                consensus_score = min(1.0, max(c["score"] for c in cl) + 0.01 * (len(cl) - 1))
                fused.append({
                    "box": [avg_x1, avg_y1, avg_x2, avg_y2],
                    "score": consensus_score,
                    "class_id": cid,
                    "fused_count": len(cl)
                })
    fused.sort(key=lambda x: x["score"], reverse=True)
    return fused

def predict_image(
    image_input,
    conf_thresh: float = 0.25,
    run_ocr: bool = True,
    enable_box_averaging: bool = True,
    enable_shrinkage: bool = True,
    shrink_factor: float = 0.96,
    fusion_iou: float = 0.50
) -> Dict[str, Any]:
    """
    Performs object detection, bounding-box recalibration (averaging + shrinkage),
    and optional PaddleOCR text recognition on an image.
    """
    if isinstance(image_input, (str, Path)):
        if not Path(image_input).exists():
            raise FileNotFoundError(f"Image not found: {image_input}")
        pil_img = Image.open(image_input).convert("RGB")
        filename = Path(image_input).name
    elif isinstance(image_input, Image.Image):
        pil_img = image_input.convert("RGB")
        filename = "in_memory_image.jpg"
    else:
        raise ValueError("image_input must be a file path or PIL Image.")
        
    orig_w, orig_h = pil_img.size
    
    # Preprocessing
    resized = pil_img.resize((RESOLUTION, RESOLUTION), Image.BILINEAR)
    x = TF.to_tensor(resized).unsqueeze(0)
    x = TF.normalize(x, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]).to(device)
    
    t0 = time.perf_counter()
    with torch.no_grad():
        if rfdetr_model is not None:
            out = rfdetr_model(x)
            pred_logits = out["pred_logits"][0] if isinstance(out, dict) else out[0][0]
            pred_boxes = out["pred_boxes"][0] if isinstance(out, dict) else out[1][0]
            scores = pred_logits.sigmoid()
        else:
            scores = torch.zeros(10, len(CLASSES), device=device)
            pred_boxes = torch.zeros(10, 4, device=device)
    det_time_ms = (time.perf_counter() - t0) * 1000
    
    max_scores, class_ids = scores.max(dim=-1)
    keep = max_scores >= conf_thresh
    
    raw_scores = max_scores[keep].cpu().numpy().tolist()
    raw_classes = class_ids[keep].cpu().numpy().tolist()
    raw_boxes_cxcywh = pred_boxes[keep].cpu().numpy()
    
    raw_boxes_xyxy = []
    for (cx, cy, w, h) in raw_boxes_cxcywh:
        x1 = float(max(0, (cx - w / 2.0) * orig_w))
        y1 = float(max(0, (cy - h / 2.0) * orig_h))
        x2 = float(min(orig_w, (cx + w / 2.0) * orig_w))
        y2 = float(min(orig_h, (cy + h / 2.0) * orig_h))
        raw_boxes_xyxy.append([x1, y1, x2, y2])
        
    # Recalibrate detections: Boundary Shrinkage & Bounding-Box Averaging
    recalibrated = recalibrate_and_average_boxes(
        boxes=raw_boxes_xyxy,
        scores=raw_scores,
        class_ids=raw_classes,
        enable_averaging=enable_box_averaging,
        enable_shrinkage=enable_shrinkage,
        shrink_factor=shrink_factor,
        fusion_iou=fusion_iou,
        max_w=orig_w,
        max_h=orig_h
    )
    
    detections = []
    t_ocr_start = time.perf_counter()
    
    for item in recalibrated:
        sc = item["score"]
        cid = item["class_id"]
        x1, y1, x2, y2 = item["box"]
        cname = CLASSES[cid] if cid < len(CLASSES) else f"class_{cid}"
        
        # Run PaddleOCR on tightly recalibrated crop
        ocr_text, ocr_conf = None, None
        if run_ocr and ocr_engine is not None and (x2 - x1 >= 6) and (y2 - y1 >= 6):
            crop = pil_img.crop((int(x1), int(y1), int(x2), int(y2)))
            crop_np = np.array(crop)
            try:
                ocr_out = ocr_engine.ocr(crop_np, cls=True)
                if ocr_out and ocr_out[0]:
                    txts = [line[1][0].strip() for line in ocr_out[0] if line[1][0].strip()]
                    cfs = [float(line[1][1]) for line in ocr_out[0]]
                    if txts:
                        ocr_text = " ".join(txts)
                        ocr_conf = round(float(np.mean(cfs)), 4)
            except Exception:
                pass
                
        detections.append({
            "class_name": cname,
            "class_id": int(cid),
            "confidence": round(float(sc), 4),
            "box_xyxy": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)],
            "fused_count": item.get("fused_count", 1),
            "ocr_text": ocr_text,
            "ocr_confidence": ocr_conf
        })
        
    ocr_time_ms = ((time.perf_counter() - t_ocr_start) * 1000) if run_ocr else 0.0
    
    return {
        "filename": filename,
        "image_width": orig_w,
        "image_height": orig_h,
        "inference_time_ms": round(det_time_ms, 2),
        "ocr_time_ms": round(ocr_time_ms, 2),
        "detections_count": len(detections),
        "detections": detections,
        "_pil_image": pil_img
    }

print("✓ predict_image function defined with Bounding-Box Averaging & Shrinkage Recalibration.")


In [ ]:
# CELL 5: Run Batch Inference on Test Samples
sample_image_candidates = []
for p in [
    REPO_ROOT / "03_dataset_split" / "dataset_stratified" / "test",
    REPO_ROOT / "coco_files" / "images",
    REPO_ROOT / "images",
    REPO_ROOT
]:
    if p.exists():
        found = list(p.glob("*.jpg")) + list(p.glob("*.png"))
        if found:
            sample_image_candidates.extend(found[:5])
            break

if not sample_image_candidates:
    print("Creating synthetic test image with shelf tag text...")
    test_img = Image.new("RGB", (800, 600), color=(225, 225, 225))
    d = ImageDraw.Draw(test_img)
    d.rectangle([150, 180, 450, 360], fill=(245, 245, 245), outline=(0, 0, 0), width=4)
    d.text((180, 240), "LOC-A-12-04", fill=(0, 0, 0))
    synthetic_path = OUTPUT_DIR / "sample_shelf.jpg"
    test_img.save(synthetic_path)
    sample_image_candidates = [synthetic_path]

print(f"Running predictions on {len(sample_image_candidates)} sample images...")
all_results = []
for img_p in sample_image_candidates:
    res = predict_image(img_p, conf_thresh=0.20, run_ocr=True)
    all_results.append(res)
    print(f"✓ {res['filename']}: {res['detections_count']} tags in {res['inference_time_ms']} ms (OCR: {res['ocr_time_ms']} ms)")
    for d in res["detections"]:
        print(f"    - [{d['class_name']}] conf={d['confidence']:.2f} text='{d['ocr_text']}' box={d['box_xyxy']}")

# Save JSON summary (without internal pil object)
export_json = [{k: v for k, v in r.items() if k != "_pil_image"} for r in all_results]
with open(OUTPUT_DIR / "predictions_summary.json", "w") as f:
    json.dump(export_json, f, indent=2)

print(f"✓ Saved prediction results to {OUTPUT_DIR / 'predictions_summary.json'}")

In [ ]:
# CELL 6: Visual Rendering with Bounding Boxes & Text Badges

colors = ["#ff3b30", "#34c759", "#007aff", "#5856d6", "#ff9500"]

for res in all_results[:3]:
    pil_img = res["_pil_image"]
    fig, ax = plt.subplots(figsize=(10, 7.5))
    ax.imshow(pil_img)
    
    for det in res["detections"]:
        x1, y1, x2, y2 = det["box_xyxy"]
        cid = det["class_id"]
        c = colors[cid % len(colors)]
        
        rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=c, linewidth=2.5)
        ax.add_patch(rect)
        
        ocr_txt = det.get("ocr_text")
        if ocr_txt:
            label = f"{det['class_name']} ({det['confidence']:.2f})\n'{ocr_txt}'"
        else:
            label = f"{det['class_name']} ({det['confidence']:.2f})"
            
        ax.text(
            x1, max(0, y1 - 8),
            label,
            color="white",
            fontsize=9,
            fontweight="bold",
            bbox=dict(facecolor=c, alpha=0.85, edgecolor="none", pad=2)
        )
        
    ax.axis("off")
    title = f"{res['filename']} ({res['detections_count']} tags detected)"
    plt.title(title, fontsize=12, pad=10)
    plt.tight_layout()
    
    save_fig_path = OUTPUT_DIR / f"predicted_{Path(res['filename']).stem}.jpg"
    plt.savefig(save_fig_path, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"✓ Saved visualization: {save_fig_path.resolve()}")

In [ ]:
# CELL 7: Summary & Transition to Evaluation
print("""
==============================================================================
 PREDICTION PIPELINE COMPLETED
 
 Results & visualizations saved to '05_prediction/predictions/'
 
 Next steps:
   - For formal benchmark metrics on clean test data:
     -> '06_evaluation/evaluate_rfdetr_clean_data.ipynb'
   - For anomaly resilience testing:
     -> '07_evaluation_anomalies/evaluate_anomalies_audit.ipynb'
   - For ONNX & AKS cloud deployment:
     -> '08_deployment/'
==============================================================================
""")